# 01 · Primeiros Passos com PySpark (Caso A)

🎯 **Objetivo:** Ganhar fluência nos quatro verbos fundamentais do PySpark — `select`, `filter`, `withColumn` e os métodos de saída `show`/`collect`/`toPandas`.

**Teoria:** docs/05-pyspark-na-pratica.md

Sem Docker — `local[*]` roda Driver e Executors como threads neste mesmo processo. Este notebook trabalha com um dataset de negócio real: `empresas`, `funcionarios` e `vendas`.

📌 O objetivo aqui **não** é entender COMO o Spark distribui trabalho (isso vem em notebooks futuros) — é ganhar prática nos verbos que você vai usar o tempo todo.

---
### 🔤 As quatro famílias de operações que você vai aprender

1. **Projeção** — `select`, `withColumn`, `drop`
2. **Filtragem** — `filter` / `where`
3. **Exploração** — `distinct`, `describe`
4. **Saída** — `show`, `collect`, `toPandas`

Todas elas se aplicam sobre DataFrame, que é a abstração central do Spark SQL.
Vamos começar criando a SparkSession.


In [ ]:
import sys

sys.path.insert(0, "../scripts")

from pyspark.sql import SparkSession

# Caso A: modo local puro. Nenhum Docker envolvido — o Spark vem embutido
# no pacote pip. master("local[*]") roda Driver E Executors como threads
# neste mesmo processo — a topologia mais simples que o Spark suporta.
spark = (
    SparkSession.builder.appName("01-primeiros-passos")
    .master("local[*]")
    # Define 8 partições para operações de shuffle (groupBy, join, etc.)
    # Em local[*] isso controla quantas tarefas rodam em paralelo
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
# Exibe a representação da SparkSession criada
spark

✅ **SparkSession criada com sucesso.** Repare na saída: o Spark exibe informações da versão (`3.5.x`), o nome da aplicação e o mestre (`local[*]`, que significa "use todos os núcleos da máquina").

📌 O `spark.sql.shuffle.partitions = 8` define em quantas partes o Spark dividirá os dados durante operações como `groupBy` e `join`. Para uma máquina local, 8 partições é um bom padrão.


## Lendo o dataset

Rode `make generate-data SCALE=small` antes, se ainda não gerou os dados.

📌 A função `layer_path` monta o caminho correto para cada camada medallion (Bronze/Silver/Gold) e cada caso de deploy (local, connect, hdfs, s3). Aqui usamos `"local"` porque estamos no Caso A.

In [ ]:
from lab_utils import layer_path

# Lê cada tabela da camada Bronze em formato Parquet (colunar, comprimido)
empresas = spark.read.parquet(layer_path("local", "bronze", "empresas"))
funcionarios = spark.read.parquet(layer_path("local", "bronze", "funcionarios"))
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))

# count() é uma AÇÃO — força o Spark a processar os dados e traz o total de linhas
# printSchema() mostra a estrutura (colunas e tipos) sem trazer todos os dados
for nome, df in [("empresas", empresas), ("funcionarios", funcionarios), ("vendas", vendas)]:
    print(f"{nome}: {df.count():,} linhas")
    df.printSchema()

📌 **Observações sobre a saída:**

- `count()` é uma **ação** — ela força o Spark a percorrer todo o dataset e computar o resultado. Sem ela, o Spark só guardaria o plano.
- `printSchema()` é uma **ação de metadados** — ela lê o esquema do Parquet (que já está embutido nos arquivos) sem precisar escanear todas as linhas.
- Repare nos tipos: `long`, `double`, `string`, `date` — cada coluna tem um tipo bem definido, diferente do Python puro.

💡 **Dica:** Parquet é o formato padrão do Spark. Ele armazena dados em colunas (não linhas), o que acelera consultas que usam poucas colunas.


## `select` e `withColumn`

`select` escolhe colunas existentes (projeção). `withColumn` cria uma **nova coluna** (ou substitui uma existente) a partir de uma expressão sobre as demais.

🧠 **Por quê?** No dia a dia você vai: (1) selecionar apenas as colunas relevantes para reduzir o volume de dados, e (2) derivar novas colunas com transformações de negócio.

💡 **Exemplo:** Calcular o salário anual a partir do salário mensal.

In [ ]:
from pyspark.sql.functions import col

# Pipeline de transformação: select -> withColumn
# Nada é executado ainda — só montamos o plano lógico (lazy evaluation)
funcionarios_resumo = (
    funcionarios
    # Projeta apenas 3 colunas das dezenas disponíveis
    .select("nome_funcionario", "cargo", "salario")
    # Cria nova coluna calculada: salário mensal x 12 meses
    .withColumn("salario_anual", col("salario") * 12)
)
# show() é uma AÇÃO: dispara o processamento e imprime as primeiras 10 linhas
funcionarios_resumo.show(10)

📌 **Entendendo a saída:**

- A coluna `salario_anual` foi calculada como `salario * 12`.
- O Spark suporta operações aritméticas diretamente via `col()`.
- O pipeline `select().withColumn()` é **lento** (lazy) — nada executa até o `show()`.

⚠️ **Atenção:** `withColumn` **não modifica** o DataFrame original. Ele retorna um **novo** DataFrame. DataFrames são imutáveis no Spark.


## `filter`

Duas perguntas de negócio simples que o `filter` responde:

1. **Quais vendas foram grandes?** (> R\$500)
2. **Quais funcionários foram contratados recentemente?** (a partir de 2024-07-01)

🧠 **Por quê?** `filter` (ou `where` — são sinônimos) é a operação mais usada para reduzir dados a apenas o que interessa para a análise. O Spark aplica o filtro o mais cedo possível no plano de execução (**predicate pushdown**) para minimizar o tráfego de dados.

In [ ]:
# Filtra vendas com valor superior a R$ 500
vendas_grandes = vendas.filter(col("valor") > 500)
# count() executa o plano e retorna o total de linhas filtradas
print(f"Vendas acima de R$500: {vendas_grandes.count():,} de {vendas.count():,}")

# Filtra funcionários admitidos a partir de julho de 2024
contratados_recentes = funcionarios.filter(col("data_admissao") >= "2024-07-01")
print(f"Funcionários admitidos desde 2024-07-01: {contratados_recentes.count():,}")
# Encadeia filter com select e show para ver apenas colunas relevantes
contratados_recentes.select("nome_funcionario", "cargo", "data_admissao").show(10)

📌 **Interpretação dos resultados:**

- A primeira contagem mostra quantas vendas são consideradas "grandes" (> R\$500) em relação ao total.
- A segunda mostra quantos funcionários foram contratados recentemente.
- Perceba como `filter().select().show()` encadeia três operações em uma linha — isso é o **estilo funcional** do Spark.

💡 **Dica:** Você pode usar `where()` no lugar de `filter()` — é um alias, o comportamento é idêntico.


## `drop`, `distinct`, `describe`

Três operações úteis para explorar e limpar dados:

- `distinct()` — remove linhas duplicadas (útil para descobrir valores únicos)
- `describe()` — estatísticas descritivas (count, mean, stddev, min, max)
- `drop()` — remove uma coluna do DataFrame

💡 **Dica:** Use `distinct()` em colunas categóricas para entender a cardinalidade dos seus dados antes de fazer agregações.

In [ ]:
# Valores únicos de setor, ordenados alfabeticamente
# truncate=False garante que textos longos não sejam cortados
setores_unicos = empresas.select("setor").distinct().orderBy("setor")
setores_unicos.show(truncate=False)

# Cargos únicos — quantos cargos diferentes existem na empresa?
cargos_unicos = funcionarios.select("cargo").distinct()
cargos_unicos.show(truncate=False)

# Estatísticas descritivas da coluna valor (média, desvio, min, max)
vendas.select("valor").describe().show()

# Remove a coluna id_funcionario do DataFrame (não altera o original)
funcionarios_sem_id = funcionarios.drop("id_funcionario")
funcionarios_sem_id.show(3)

📌 **Resumo das operações:**

- `distinct()` revelou quantos setores e cargos diferentes existem nos dados — informação valiosa antes de fazer agregações.
- `describe()` forneceu média, desvio padrão, mínimo e máximo da coluna `valor` — um resumo estatístico rápido.
- `drop()` removeu a coluna `id_funcionario` sem alterar o DataFrame original.

🧠 **Por quê** `drop` não altera o original? DataFrames Spark são **imutáveis**. Toda transformação retorna um novo DataFrame.


## `show` vs. `collect` vs. `toPandas`

Cada método de saída tem um propósito e um custo diferente:

| Método | Retorno | Quando usar |
|---|---|---|
| `show()` | `None` (só imprime) | Dar uma olhada rápida nos dados |
| `collect()` | `list[Row]` | Pequenos resultados para processar em Python puro |
| `toPandas()` | `pd.DataFrame` | Plotar, exportar ou integrar com Pandas/ML |

⚠️ **Atenção:** `collect()` e `toPandas()` trazem **todos os dados para a memória do Driver**. Com datasets grandes, isso pode causar `OutOfMemoryError`. Sempre use `.limit()` ou `.filter()` antes.

In [ ]:
# Pega apenas 5 linhas como amostra (evita trazer tudo para o Driver)
amostra = empresas.limit(5)

# show(): imprime na tela, não retorna nada utilizável
amostra.show()
# collect(): retorna uma lista Python de objetos Row
linhas = amostra.collect()
print(type(linhas), linhas[0])

# toPandas(): converte para pandas DataFrame (requer que os dados caibam na RAM)
pdf = amostra.toPandas()
print(type(pdf))
# Exibe o pandas DataFrame com formatação rica
pdf

📌 **Quando usar cada um?**

- **`show()`** — uso diário para dar uma espiada nos dados
- **`collect()`** — bom para resultados pequenos que você quer processar com lógica Python (ex: loops, condicionais)
- **`toPandas()`** — ideal para gerar gráficos com matplotlib/seaborn ou exportar para CSV/Excel

⚠️ **Regra de ouro:** Nunca faça `collect()` ou `toPandas()` em um DataFrame inteiro sem antes filtrar ou limitar. Sempre pergunte: *quanto disso cabe na RAM do meu laptop?*


In [ ]:
# Encerra a SparkSession e libera recursos (threads, memória)
# Bom prática: sempre parar a sessão ao final do notebook
spark.stop()

---
🎉 **Parabéns!** Você completou o primeiro notebook de PySpark.

Você aprendeu:
- Criar uma SparkSession
- Ler arquivos Parquet
- `select`, `withColumn`, `filter`, `drop`, `distinct`, `describe`
- A diferença entre `show`, `collect` e `toPandas`
- O conceito de **lazy evaluation** (transformações vs. ações)

▶️ **Próximo:** Notebook 02 — Agregações de Negócio com `groupBy`/`agg`
